# Freight Rate Prediction Challenge - Google Colab Walkthrough

This notebook follows the assessment requirements end to end:

1. Prepare the supplied files.
2. Explore data and data-quality issues.
3. Use a chronological validation split that mimics future prediction.
4. Train a CatBoost ensemble.
5. Create `validation_predictions.csv`.
6. Fill `data/december_chart_inputs.csv`.
7. Run the supplied scorer and generate `scorer_results/candidate_december.png`.

**Recommended:** upload the provided `freight_rate_solution.zip` to Colab, then execute the cells in order.

## 1. Upload and unzip the project

In Colab, upload `freight_rate_solution.zip` from the Files pane, then run the next cell. If you instead upload the raw assessment files, make sure they are renamed to the underscore-style filenames shown in `data/`.

In [ ]:
# If the ZIP is already in /content, this cell extracts it.
from pathlib import Path
import os

zip_path = Path('/content/freight_rate_solution.zip')
if zip_path.exists():
    !rm -rf /content/freight_rate_solution
    !unzip -q /content/freight_rate_solution.zip -d /content/

PROJECT = Path('/content/freight_rate_solution')
assert PROJECT.exists(), 'Upload freight_rate_solution.zip first.'
os.chdir(PROJECT)
print('Working directory:', Path.cwd())

## 2. Install dependencies

The original scorer only needs pandas/numpy/matplotlib. The ML solution additionally uses CatBoost and scikit-learn.

In [ ]:
!pip -q install -r requirements.txt

## 3. Load and inspect the datasets

The labeled development set contains the target `posted_rate`. The final validation set has the same load features but no target.

In [ ]:
import pandas as pd

train = pd.read_csv('data/train_test.csv')
validation = pd.read_csv('data/validation.csv')
template = pd.read_csv('data/validation_predictions_template.csv')
december = pd.read_csv('data/december_chart_inputs.csv')

print('train:', train.shape)
print('validation:', validation.shape)
print('template:', template.shape)
print('december:', december.shape)
display(train.head())

## 4. Data-quality audit

Important findings from the supplied data:

- Training has missing `weight` and `market_index` values.
- Final validation also has missing `weight` and `market_index` values.
- The final validation period is later in time than the labeled data.
- Some cities in validation are unseen during training.

These points are why a chronological split and a model with native missing/categorical handling are appropriate.

In [ ]:
train['date'] = pd.to_datetime(train['date'])
validation['date'] = pd.to_datetime(validation['date'])

print('Train date range:', train['date'].min(), 'to', train['date'].max())
print('Validation date range:', validation['date'].min(), 'to', validation['date'].max())
print('
Train missing values:')
display(train.isna().sum()[train.isna().sum() > 0].to_frame('missing'))
print('
Validation missing values:')
display(validation.isna().sum()[validation.isna().sum() > 0].to_frame('missing'))

unseen_pickups = sorted(set(validation['pickup']) - set(train['pickup']))
unseen_deliveries = sorted(set(validation['delivery']) - set(train['delivery']))
print('Unseen pickup cities:', unseen_pickups)
print('Unseen delivery cities:', unseen_deliveries)

## 5. Run the complete training and prediction pipeline

The script uses:

- January-August 2025 as development training data.
- September-October 2025 as a future-style holdout.
- Route, calendar, cyclical date, geographic, distance, weight, market and quote features.
- A three-seed CatBoost ensemble.
- A reduced-feature CatBoost model for the fixed December chart because the December file does not contain coordinates, `market_index`, or `quote_signal`.

In [ ]:
!python src/train_predict.py --project-dir . --run-scorer

## 6. Review the validation metrics and generated files

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path('outputs/run_summary.json').read_text())
print(json.dumps(summary['baseline_metrics'], indent=2))
print('
CatBoost temporal holdout metrics:')
print(json.dumps(summary['catboost_ensemble_metrics'], indent=2))

preds = pd.read_csv('validation_predictions.csv')
dec = pd.read_csv('data/december_chart_inputs.csv')
print('
Prediction file shape:', preds.shape)
print('Prediction columns:', list(preds.columns))
display(preds.head())
display(dec.head())

## 7. Display the scorer-generated December chart

In [ ]:
from IPython.display import Image, display

display(Image(filename='scorer_results/candidate_december.png'))

## 8. Final structural checks

The assessment requires exactly 12,000 rows with columns `load_id,predicted_rate`, and the scorer also checks that all predicted rates are positive.

In [ ]:
assert list(preds.columns) == ['load_id', 'predicted_rate']
assert len(preds) == 12_000
assert preds['load_id'].is_unique
assert preds['predicted_rate'].notna().all()
assert (preds['predicted_rate'] > 0).all()
assert len(dec) == 31
assert dec['predicted_rate'].notna().all()
assert (dec['predicted_rate'] > 0).all()
print('All final checks passed.')

## 9. Download the required submission files

In [ ]:
from google.colab import files

files.download('validation_predictions.csv')
files.download('data/december_chart_inputs.csv')
files.download('scorer_results/candidate_december.png')
# The report PDF is already included in the project under report/.